In [12]:
# ============================================================================
# 01_clean_and_standardize.ipynb
# Pipeline: RAW → Limpeza → Padronização Colunas → Padronização Países
# 
# VERSÃO COM MÓDULO CENTRAL (mapeamento_paises.py)
# ============================================================================

In [13]:
#import os, sys
#from pathlib import Path

#print("CWD:", os.getcwd())
#print("\nArquivos na raiz:")
#print(os.listdir('.'))
#print("\nsys.path:")
#for p in sys.path:
#    print(" ", p)

In [14]:
# ============================================================================
# Célula 1: Imports e Configuração
# ============================================================================
import pandas as pd
import glob
import os
import re
import unicodedata
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Imports do módulo central
from mapeamento import padronizar_colunas, MAPA_GLOBAL_COLUNAS
from mapeamento_paises import (
    DICT_CUSTOM, 
    MAPA_ISO_PT, 
    INDICE_CANONICO,
    NOMES_CANONICOS_PT,
    validar_mapeamentos
)

In [15]:
# ============================================================================
# Célula 2: Configuração dos Caminhos
# ============================================================================
PASTA_ORIGEM = r"C:\zardit\personal-course\data\raw\aima"
PASTA_DESTINO = r"C:\zardit\personal-course\data\clean\aima"
PASTA_PADRONIZADO = os.path.join(PASTA_DESTINO, "padronizado")

# Cria pastas
for pasta in [PASTA_DESTINO, PASTA_PADRONIZADO]:
    os.makedirs(pasta, exist_ok=True)

# Configurações
COLUNAS_PARA_EXCLUIR = ['Stock_Total', 'Fluxos_Total', 'Pop_Residente_Total', 'Concessao_Total']
SEPARADOR = ';'
ENCODING = 'utf-8'
FUZZY_THRESHOLD = 82

print("✅ Configuração carregada")
print(f"📁 Origem: {PASTA_ORIGEM}")
print(f"📁 Destino: {PASTA_DESTINO}")

✅ Configuração carregada
📁 Origem: C:\zardit\personal-course\data\raw\aima
📁 Destino: C:\zardit\personal-course\data\clean\aima


In [16]:
# ============================================================================
# Célula 3: Validação dos Mapeamentos
# ============================================================================
validar_mapeamentos()

🔍 Validando mapeamentos de países...
   DICT_CUSTOM: 90 entradas
   MAPA_ISO_PT: 249 entradas
   NOMES_CANONICOS: 257 nomes únicos
   ⚠️ Valores duplicados no DICT_CUSTOM: ['São Tomé e Príncipe', 'Ilhas Fiji', 'Guiné Equatorial', 'Coreia do Norte', 'Macedónia do Norte', 'Reino Unido', 'Rússia', 'Coreia do Sul', 'Países Baixos', 'Ilhas Maurícias', 'Costa do Marfim', 'Estados Unidos da América', 'Guiné-Bissau', 'República Checa', 'Cabo Verde', 'África do Sul', 'Turquia', 'Essuatíni', 'Timor-Leste', 'Mianmar']
✅ Validação concluída


True

In [17]:
# ============================================================================
# Célula 4: Funções de Limpeza
# ============================================================================
def limpar_e_padronizar_dataframe(df):
    """
    Aplica toda a pipeline de limpeza em um DataFrame
    """
    # 1. Remove colunas de total
    cols_to_drop = [col for col in COLUNAS_PARA_EXCLUIR if col in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"   🗑️ Removidas colunas: {cols_to_drop}")
    
    # 2. Padroniza nomes das colunas (PT → EN snake_case)
    df = padronizar_colunas(df)
    
    # 3. Remove linhas indesejadas (totais, cabeçalhos)
    if 'nationality' in df.columns:
        df = df[df['nationality'].notna()]
        df = df[~df['nationality'].str.contains('TOTAL|Total|Nacionalidade', na=False, case=False)]
        df = df.dropna(how='all')
    
    # 4. Converte colunas numéricas
    numeric_cols = [col for col in df.columns if col != 'nationality']
    for col in numeric_cols:
        # Remove pontos de milhares
        df[col] = df[col].astype(str).str.replace('.', '', regex=False)
        # Converte para numérico
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Substitui 0 por NA (opcional)
        df[col] = df[col].replace(0, pd.NA)
        # Converte para Int64 (suporta NA)
        df[col] = df[col].astype('Int64')
    
    return df

In [18]:
# ============================================================================
# Célula 5: Processamento em Lote (Limpeza + Padronização Colunas)
# ============================================================================
# Busca arquivos
input_files = glob.glob(os.path.join(PASTA_ORIGEM, "rma_*.csv"))
if not input_files:
    # Tenta outro padrão se necessário
    input_files = glob.glob(os.path.join(PASTA_ORIGEM, "*.csv"))

print(f"📁 Encontrados {len(input_files)} arquivos para processar\n")

arquivos_limpos = []
for file_path in input_files:
    nome_arquivo = os.path.basename(file_path)
    print(f"🔄 Processando: {nome_arquivo}")
    
    # Carrega e limpa
    df = pd.read_csv(file_path, encoding=ENCODING)
    df_clean = limpar_e_padronizar_dataframe(df)
    
    # Salva versão limpa (antes da padronização de países)
    output_file = os.path.join(PASTA_DESTINO, f"{nome_arquivo}")
    df_clean.to_csv(output_file, sep=SEPARADOR, encoding=ENCODING, index=False, na_rep='')
    arquivos_limpos.append(output_file)
    print(f"   ✅ Salvo: {os.path.basename(output_file)} ({len(df_clean)} linhas)\n")

print(f"\n✅ {len(arquivos_limpos)} arquivos limpos e salvos!")

# ============================================================================
# Célula 6: Configuração para Padronização de Países
# ============================================================================
# Usa os mapeamentos importados de mapeamento_paises.py
# Nada mais a configurar - tudo já está no módulo central!

📁 Encontrados 2 arquivos para processar

🔄 Processando: rma_2023_pg36_residents_by_nationality_gender.csv
   🗑️ Removidas colunas: ['Pop_Residente_Total', 'Concessao_Total']
   ✅ Salvo: rma_2023_pg36_residents_by_nationality_gender.csv (192 linhas)

🔄 Processando: rma_2024_pg37_residents_by_nationality_gender.csv
   🗑️ Removidas colunas: ['Pop_Residente_Total', 'Concessao_Total']
   ✅ Salvo: rma_2024_pg37_residents_by_nationality_gender.csv (197 linhas)


✅ 2 arquivos limpos e salvos!


In [23]:
# ============================================================================
# Célula 7: Funções de Normalização de Países
# ============================================================================
import country_converter as coco
from rapidfuzz import process, fuzz

_CC = coco.CountryConverter()

def _strip(text: str) -> str:
    """Normaliza para comparação: lowercase + colapsa espaços."""
    return re.sub(r"\s+", " ", str(text).strip().lower())

def _strip_accents(text: str) -> str:
    """Remove acentos para comparação fuzzy mais tolerante."""
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

def normalizar_pais(nome):
    """
    Pipeline completo de normalização de nomes de países
    Usa os mapeamentos do módulo central
    """
    if not nome or not isinstance(nome, str):
        return nome, "nulo", 0.0
    
    nome_norm = _strip(nome)
    
    # 1. Verifica se já é canônico (O(1) lookup)
    if nome_norm in INDICE_CANONICO:
        return INDICE_CANONICO[nome_norm], "canonico", 100.0
    
    # 2. Dicionário custom
    if nome_norm in DICT_CUSTOM:
        return DICT_CUSTOM[nome_norm], "custom", 100.0
    
    # 3. Country Converter (usando MAPA_ISO_PT do módulo central)
    try:
        iso2 = _CC.convert(names=nome, to="ISO2")
        if iso2 and iso2 not in ["not_found", "not found"]:
            pt_name = MAPA_ISO_PT.get(iso2)
            if pt_name:
                return pt_name, "coco", 100.0
    except Exception:
        pass
    
    # 4. Fuzzy matching
    # Usa NOMES_CANONICOS_PT do módulo central
    candidates = list(DICT_CUSTOM.keys()) + list(NOMES_CANONICOS_PT)
    clean_map = {_strip_accents(_strip(c)): c for c in candidates}
    
    match = process.extractOne(
        _strip_accents(nome_norm),
        list(clean_map.keys()),
        scorer=fuzz.ratio,
        score_cutoff=FUZZY_THRESHOLD,
    )
    
    if match:
        original_key = clean_map[match[0]]
        resultado = DICT_CUSTOM.get(original_key, original_key)
        return resultado, f"fuzzy({match[1]:.0f})", match[1]
    
    return nome, "nao_resolvido", 0.0

In [25]:
# ============================================================================
# Célula 8: Aplicar Padronização de Países em Todos os CSVs
# ============================================================================
# Busca os arquivos já limpos
clean_files = glob.glob(os.path.join(PASTA_DESTINO, "rma_*.csv"))

if not clean_files:
    print("⚠️ Nenhum arquivo limpo encontrado!")
else:
    print(f"\n🌍 Iniciando padronização de países em {len(clean_files)} arquivos...\n")
    
    registro_conversoes = []
    nao_resolvidos = []
    
    for file_path in clean_files:
        nome_arquivo = os.path.basename(file_path)
        print(f"🌍 Padronizando países: {nome_arquivo}")
        
        df = pd.read_csv(file_path, sep=SEPARADOR, encoding=ENCODING)
        
        numeric_cols = [c for c in df.columns if c != 'nationality']
        for col in numeric_cols:
            df[col] = df[col].astype('Int64')
        
        if 'nationality' not in df.columns:
            print(f"   ⚠️ Coluna 'nationality' não encontrada em {nome_arquivo}")
            continue
        
        # Aplica normalização
        resultados = df['nationality'].fillna('').apply(normalizar_pais)
        df['nationality'] = resultados.apply(lambda x: x[0])
        df['_metodo'] = resultados.apply(lambda x: x[1])
        df['_confianca'] = resultados.apply(lambda x: x[2])
        
        # Registra conversões
        for idx, row in df.iterrows():
            metodo = row['_metodo']
            if metodo == 'nao_resolvido':
                nao_resolvidos.append({
                    'arquivo': nome_arquivo,
                    'original': row['nationality'],
                    'metodo': metodo
                })
            registro_conversoes.append({
                'arquivo': nome_arquivo,
                'original': row['nationality'],
                'metodo': metodo,
                'confianca': row['_confianca']
            })
        
        # Remove colunas auxiliares e salva
        df_sem_aux = df.drop(columns=['_metodo', '_confianca'])
        output_path = os.path.join(PASTA_PADRONIZADO, nome_arquivo)
        df_sem_aux.to_csv(output_path, sep=SEPARADOR, encoding=ENCODING, index=False)
        print(f"   ✅ Salvo em: {os.path.basename(output_path)} ({len(df)} linhas)\n")


🌍 Iniciando padronização de países em 2 arquivos...

🌍 Padronizando países: rma_2023_pg36_residents_by_nationality_gender.csv
   ✅ Salvo em: rma_2023_pg36_residents_by_nationality_gender.csv (192 linhas)

🌍 Padronizando países: rma_2024_pg37_residents_by_nationality_gender.csv
   ✅ Salvo em: rma_2024_pg37_residents_by_nationality_gender.csv (197 linhas)



In [26]:
# ============================================================================
# Célula 9: Relatório de Padronização
# ============================================================================
if registro_conversoes:
    df_registro = pd.DataFrame(registro_conversoes)
    
    print("\n" + "="*60)
    print("📊 RELATÓRIO DE PADRONIZAÇÃO DE PAÍSES")
    print("="*60)
    
    print("\n📈 Distribuição por método:")
    print(df_registro['metodo'].value_counts().to_string())
    
    # Análise de confiança para fuzzy
    fuzzy_cases = df_registro[df_registro['metodo'].str.startswith('fuzzy', na=False)]
    if not fuzzy_cases.empty:
        # Extrai scores
        fuzzy_cases['score'] = fuzzy_cases['metodo'].str.extract(r'(\d+)').astype(float)
        print(f"\n🎯 Estatísticas Fuzzy (n={len(fuzzy_cases)}):")
        print(f"   Min Score: {fuzzy_cases['score'].min():.0f}")
        print(f"   Max Score: {fuzzy_cases['score'].max():.0f}")
        print(f"   Média: {fuzzy_cases['score'].mean():.1f}")
    
    # Casos não resolvidos
    if nao_resolvidos:
        df_nr = pd.DataFrame(nao_resolvidos).drop_duplicates()
        print(f"\n⚠️ {len(df_nr)} CASOS NÃO RESOLVIDOS ENCONTRADOS!")
        print("\n👉 Adicione ao DICT_CUSTOM em 'scripts/mapeamento_paises.py':")
        print(df_nr[['original']].drop_duplicates().to_string(index=False))
        
        # Salva lista para revisão
        nr_path = os.path.join(PASTA_PADRONIZADO, "nao_resolvidos.csv")
        df_nr.to_csv(nr_path, sep=SEPARADOR, encoding=ENCODING, index=False)
        print(f"\n   💾 Lista salva em: {nr_path}")
    else:
        print("\n🎉 TODOS OS PAÍSES FORAM RESOLVIDOS AUTOMATICAMENTE!")
    
    # Resumo por arquivo
    print("\n📄 Resumo por arquivo:")
    resumo = df_registro.groupby(['arquivo', 'metodo']).size().unstack(fill_value=0)
    resumo['total'] = resumo.sum(axis=1)
    print(resumo.to_string())

print("\n" + "="*60)
print("✅ Notebook 1 concluído com sucesso!")
print("="*60)


📊 RELATÓRIO DE PADRONIZAÇÃO DE PAÍSES

📈 Distribuição por método:
metodo
canonico    351
custom       25
coco         13

🎉 TODOS OS PAÍSES FORAM RESOLVIDOS AUTOMATICAMENTE!

📄 Resumo por arquivo:
metodo                                             canonico  coco  custom  total
arquivo                                                                         
rma_2023_pg36_residents_by_nationality_gender.csv       180     4       8    192
rma_2024_pg37_residents_by_nationality_gender.csv       171     9      17    197

✅ Notebook 1 concluído com sucesso!


In [28]:
# ============================================================================
# Célula 10: Relatório de Limpeza por Arquivo (TXT)
# Lê os ficheiros raw/clean/padronizado do disco e gera um bloco por arquivo.
# Não modifica nenhuma célula anterior.
# ============================================================================
from datetime import datetime

def _sep(char='─', n=80): return char * n

_r1 = []          # buffer de linhas do relatório
_add = _r1.append # atalho

_add(_sep('='))
_add('RELATÓRIO DE LIMPEZA DE DADOS — NOTEBOOK 1')
_add(f'Data: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
_add('Pipeline: RAW → Limpeza → Padronização Colunas → Padronização Países')
_add(_sep('='))

# Ficheiros raw a processar
_raw_files = sorted(glob.glob(os.path.join(PASTA_ORIGEM, 'rma_*.csv')))

# DataFrame com registo de conversões de países (definido na Célula 8)
_df_reg = pd.DataFrame(registro_conversoes) if registro_conversoes else pd.DataFrame()

_totais = dict(arqs=0, raw=0, clean=0, pad=0)

for _raw_path in _raw_files:
    _nome      = os.path.basename(_raw_path)
    _clean_path = os.path.join(PASTA_DESTINO, _nome)
    _pad_path   = os.path.join(PASTA_PADRONIZADO, _nome)

    # ── Lê raw ──────────────────────────────────────────────────────────────
    _df_raw   = pd.read_csv(_raw_path, encoding=ENCODING)
    _cols_raw = _df_raw.columns.tolist()
    _n_raw    = len(_df_raw)

    # ── Lê clean (pós-limpeza, pré-países) ──────────────────────────────────
    if os.path.exists(_clean_path):
        _df_cl   = pd.read_csv(_clean_path, sep=SEPARADOR, encoding=ENCODING)
        _cols_cl = _df_cl.columns.tolist()
        _n_cl    = len(_df_cl)
    else:
        _cols_cl, _n_cl = [], 0

    # ── Lê padronizado (pós-países) ─────────────────────────────────────────
    if os.path.exists(_pad_path):
        _df_pad = pd.read_csv(_pad_path, sep=SEPARADOR, encoding=ENCODING)
        _n_pad  = len(_df_pad)
    else:
        _df_pad, _n_pad = None, 0

    # ── Calcula transformações aplicadas ────────────────────────────────────
    _excluidas = [c for c in COLUNAS_PARA_EXCLUIR if c in _cols_raw]
    _unnamed   = [c for c in _cols_raw if str(c).startswith('Unnamed')]
    _removidas = _excluidas + _unnamed

    # Colunas renomeadas: as que estão no mapa e não foram removidas
    _renomeadas = {
        c: MAPA_GLOBAL_COLUNAS[c]
        for c in _cols_raw
        if c in MAPA_GLOBAL_COLUNAS and c not in _removidas
    }

    _n_rem = _n_raw - _n_cl
    _pct   = (_n_rem / _n_raw * 100) if _n_raw else 0.0

    # Métodos de padronização de países para este ficheiro
    _metodos = {}
    if not _df_reg.empty and 'arquivo' in _df_reg.columns:
        _metodos = _df_reg[_df_reg['arquivo'] == _nome]['metodo'].value_counts().to_dict()

    # ── Bloco do ficheiro ───────────────────────────────────────────────────
    _add('')
    _add(_sep())
    _add(f'📄 ARQUIVO: {_nome}')
    _add(_sep())
    _add(f"  {'Linhas originais':<40} {_n_raw:>6,}")
    _add(f"  {'Linhas após limpeza':<40} {_n_cl:>6,}   "
         f"(removidas: {_n_rem:,} | {_pct:.1f}%)")
    _add(f"  {'Linhas após padronização países':<40} {_n_pad:>6,}")
    _add(f"  {'Colunas originais':<40} {len(_cols_raw):>6}")
    _add(f"  {'Colunas finais':<40} {len(_cols_cl):>6}")

    # Colunas removidas
    if _removidas:
        _add(f"\n  🗑️  Colunas removidas ({len(_removidas)}):")
        for _c in _excluidas:
            _add(f"       - {_c:<38}  # coluna de total")
        for _c in _unnamed:
            _add(f"       - {_c:<38}  # sem nome (artefacto Excel)")

    # Colunas renomeadas
    if _renomeadas:
        _add(f"\n  🔤 Colunas renomeadas ({len(_renomeadas)}) [PT → EN snake_case]:")
        for _orig, _novo in _renomeadas.items():
            _add(f"       {_orig:<38} → {_novo}")

    # Padronização de países
    _add("\n  🌍 Padronização de países:")
    if _metodos:
        for _met, _cnt in sorted(_metodos.items(), key=lambda x: -x[1]):
            _add(f"       {_met:<28} {_cnt:>4} registos")
    else:
        _add("       (sem dados de conversão nesta execução)")

    # Estatísticas numéricas
    if _df_pad is not None:
        _add("\n  📊 Estatísticas numéricas (pós-padronização):")
        for _col in _df_pad.select_dtypes(include='number').columns:
            _soma  = _df_pad[_col].sum()
            _nulos = int(_df_pad[_col].isna().sum())
            _add(f"       {_col:<38}  soma: {_soma:>12,.0f}   nulos: {_nulos:>4}")

    _totais['arqs']  += 1
    _totais['raw']   += _n_raw
    _totais['clean'] += _n_cl
    _totais['pad']   += _n_pad

# ── Sumário geral ──────────────────────────────────────────────────────────
_add('')
_add(_sep('='))
_add('📊 SUMÁRIO GERAL')
_add(_sep())
_add(f"  {'Arquivos processados':<40} {_totais['arqs']}")
_add(f"  {'Total linhas originais':<40} {_totais['raw']:,}")
_add(f"  {'Total linhas após limpeza':<40} {_totais['clean']:,}")
_add(f"  {'Total linhas padronizadas':<40} {_totais['pad']:,}")
_total_rem  = _totais['raw'] - _totais['clean']
_pct_total  = (_total_rem / _totais['raw'] * 100) if _totais['raw'] else 0
_add(f"  {'Total linhas removidas':<40} {_total_rem:,}  ({_pct_total:.1f}%)")

if not _df_reg.empty:
    _add("\n  🌍 Métodos de padronização (todos os ficheiros):")
    for _met, _cnt in _df_reg['metodo'].value_counts().items():
        _add(f"       {_met:<28} {_cnt:>5}")

_add('')
_add(_sep('='))
_add('FIM DO RELATÓRIO — NOTEBOOK 1')
_add(_sep('='))

# ── Imprime e guarda ───────────────────────────────────────────────────────
_txt_nb1 = '\n'.join(_r1)
print(_txt_nb1)

_path_nb1 = os.path.join(PASTA_PADRONIZADO, 'relatorio_limpeza_nb1.txt')
with open(_path_nb1, 'w', encoding=ENCODING) as _f:
    _f.write(_txt_nb1)
print(f'\n💾 Relatório guardado em: {_path_nb1}')

RELATÓRIO DE LIMPEZA DE DADOS — NOTEBOOK 1
Data: 2026-07-10 13:05:44
Pipeline: RAW → Limpeza → Padronização Colunas → Padronização Países

────────────────────────────────────────────────────────────────────────────────
📄 ARQUIVO: rma_2023_pg36_residents_by_nationality_gender.csv
────────────────────────────────────────────────────────────────────────────────
  Linhas originais                            192
  Linhas após limpeza                         192   (removidas: 0 | 0.0%)
  Linhas após padronização países             192
  Colunas originais                             7
  Colunas finais                                5

  🗑️  Colunas removidas (2):
       - Pop_Residente_Total                     # coluna de total
       - Concessao_Total                         # coluna de total

  🔤 Colunas renomeadas (5) [PT → EN snake_case]:
       Nacionalidade                          → nationality
       Pop_Residente_Masculino                → resident_count_male
       Pop_Residente_F